# Evaluación RAG v2 — Qwen2.5-7B-Instruct | Dataset evaluacion_completa | Enfoque RAG v3
**Modelo:** Qwen2.5-7B-Instruct (4bit) + QLoRA opcional  
**Dataset:** `evaluacion_completa.csv` (columnas `Requirement`, `Answer`; `Source_chunks` NO se usa)  
**Corpus RAG:** `cisco_ios_xe_16.11.1_chunks.jsonl`  
**Diferencias vs topo_v1:**
- Normalización de intent: texto técnico simple para query RAG
- Query RAG: `Normalized intent + Original requirement`
- Retrieval: `top_k=3` sobre índice FAISS prefiltrado por arquitectura
- Contexto recuperado: formato verbose `[DOCUMENTATION CHUNK i]` con OS/Version
- Prompt de generación: estilo v3 con `configure terminal`/`end` y opción `NO_CODE`

**Métricas:** ROUGE-1/2/L normalizado (sin prompts CLI) + BERTScore | **GPU:** T4

---
### Checklist antes de ejecutar
1. Menú → **Entorno de ejecución → Cambiar tipo de entorno** → GPU T4  
2. Sube a la carpeta `dataset eval` de tu Drive (`/content/drive/MyDrive/dataset eval`): `evaluacion_completa.csv` y `cisco_ios_xe_16.11.1_chunks.jsonl`  
3. Agrega tu `HF_TOKEN` en Secrets (ícono 🔑)  
4. Ajusta los flags en **Celda 2 — Configuración** (incluida `DRIVE_BASE` si tu carpeta tiene otro nombre/ubicación)

## 📦 Celda 1 — Instalación de dependencias

In [ ]:
!pip install -q transformers==4.46.3 peft accelerate bitsandbytes sentencepiece rouge-score==0.1.2 bert-score==0.3.13 sentence-transformers faiss-cpu

print('✅ Dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 120.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ Dependencias instaladas


## ⚙️ Celda 2 — Configuración

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Carpeta en Drive donde estan el notebook y los tres archivos de entrada.
'''
DRIVE_BASE         = '/content/drive/MyDrive'
DRIVE_DATASET_PATH = DRIVE_BASE + '/base + rag v 17/dataset evaluacion v17/eval_dataset_150_cli_version17.csv'
DRIVE_RAG_PATH     = DRIVE_BASE + '/implementacionFinal/RAG_base_completo_137k.jsonl'
EMB_PATH           = DRIVE_BASE + '/implementacionFinal/bge/rag_embeddings_unificado_bge.npy'
META_PATH          = DRIVE_BASE + '/implementacionFinal/bge/rag_metadata_unificado_bge.parquet'
'''
DRIVE_BASE         = '/content/drive/MyDrive'
DRIVE_DATASET_PATH = DRIVE_BASE + '/base + rag v 17/dataset evaluacion v17/eval_dataset_150_cli_version17.csv'
DRIVE_RAG_PATH     = DRIVE_BASE + '/implementacionFinal/RAG_base_completo_137k.jsonl'
EMB_PATH           = DRIVE_BASE + '/implementacionFinal/mini/rag_embeddings_unificado.npy'
META_PATH          = DRIVE_BASE + '/implementacionFinal/mini/rag_metadata_unificado.parquet'

# Arquitectura documental evaluada. El filtro se aplica una sola vez al inicio.

ARCH_BASE = {
    'os': 'Cisco IOS XE',
    'version': ['17.12.x', '17.12.1', '17.x'],
    'device_type': ['router', 'switch'],
    'product': [
        '4000 Series Integrated Services Routers',
        'Catalyst 9300 Series Switches',
    ],
}
'''
ARCH_BASE = {
    'os':          'Cisco IOS XE',                             # filtro duro (exacto, normalizado)
    'version':     '16.11.x',                                  # filtro duro por familia major.minor
    'device_type': 'router',                                   # 'router' | 'switch' | 'both'
    'product':     '4000 Series Integrated Services Routers',  # filtro suave (substring)
}
'''

USE_PRODUCT_FILTER = True
USE_BATFISH_COMPAT_FILTER = True
STRICT_PRODUCT     = False
PRODUCT_MIN_KEEP   = 1
VERSION_FALLBACK   = True

USE_FINETUNED   = False
BASE_MODEL_PATH = 'Qwen/Qwen2.5-7B-Instruct'
ADAPTER_PATH    = 'felbol1/qwen25-7b-networking-v3'

USE_RAG              = True
USE_INTENT_NORMALIZATION = True
RAG_TOP_K            = 3
USE_RAG_MIN_SCORE    = True
RAG_MIN_SCORE        = 0.6
#EMBEDDING_MODEL_NAME = 'BAAI/bge-base-en-v1.5'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

MAX_NEW_TOKENS        = 512
MAX_NEW_TOKENS_INTENT = 120

_ft  = 'QLoRA' if USE_FINETUNED else 'Base'
_rag = 'RAG'   if USE_RAG       else 'NoRAG'
MODEL_NAME = 'Qwen2.5-7B-' + _ft + '+' + _rag + '_simple_rag_arch_prefilter'

RESULTS_GROUP        = 'base+rag' if USE_RAG else 'base'
DRIVE_RESULTS_FOLDER = DRIVE_BASE + '/resultados/' + RESULTS_GROUP

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME']  = '/root/.cache/huggingface'

print('Experimento          : ' + MODEL_NAME)
print('Arquitectura base    : ' + str(ARCH_BASE))
print('USE_RAG              : ' + str(USE_RAG))
print('USE_INTENT_NORMALIZATION: ' + str(USE_INTENT_NORMALIZATION))
print('USE_BATFISH_COMPAT_FILTER: ' + str(USE_BATFISH_COMPAT_FILTER))
print('USE_FINETUNED        : ' + str(USE_FINETUNED))
print('RAG_TOP_K            : ' + str(RAG_TOP_K))
print('USE_RAG_MIN_SCORE    : ' + str(USE_RAG_MIN_SCORE))
print('RAG_MIN_SCORE        : ' + str(RAG_MIN_SCORE))
print('MAX_NEW_TOKENS       : ' + str(MAX_NEW_TOKENS))
print('MAX_NEW_TOKENS_INTENT: ' + str(MAX_NEW_TOKENS_INTENT))
print('Carpeta resultados   : ' + DRIVE_RESULTS_FOLDER)


Experimento          : Qwen2.5-7B-Base+RAG_simple_rag_arch_prefilter
Arquitectura base    : {'os': 'Cisco IOS XE', 'version': ['17.12.x', '17.12.1', '17.x'], 'device_type': ['router', 'switch'], 'product': ['4000 Series Integrated Services Routers', 'Catalyst 9300 Series Switches']}
USE_RAG              : True
USE_INTENT_NORMALIZATION: True
USE_BATFISH_COMPAT_FILTER: True
USE_FINETUNED        : False
RAG_TOP_K            : 3
USE_RAG_MIN_SCORE    : True
RAG_MIN_SCORE        : 0.6
MAX_NEW_TOKENS       : 512
MAX_NEW_TOKENS_INTENT: 120
Carpeta resultados   : /content/drive/MyDrive/resultados/base+rag


## 🔌 Celda 3 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

for path in [DRIVE_DATASET_PATH, DRIVE_RAG_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError('Archivo no encontrado: ' + path)
    print('OK ' + path)

os.makedirs(DRIVE_RESULTS_FOLDER, exist_ok=True)
print('OK Carpeta de resultados: ' + DRIVE_RESULTS_FOLDER)


Mounted at /content/drive
OK /content/drive/MyDrive/base + rag v 17/dataset evaluacion v17/eval_dataset_150_cli_version17.csv
OK /content/drive/MyDrive/implementacionFinal/RAG_base_completo_137k.jsonl
OK Carpeta de resultados: /content/drive/MyDrive/resultados/base+rag


## 🖥️ Celda 4 — Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('❌ CUDA no disponible. Activa GPU en Entorno de ejecucion.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('✅ GPU: ' + gpu_name + '  |  VRAM: ' + '{:.1f}'.format(vram_gb) + ' GB')

if vram_gb < 14:
    print('⚠️  Menos de 14 GB. El modelo 4bit puede no caber.')

✅ GPU: NVIDIA L4  |  VRAM: 22.0 GB


## 📂 Celda 5 — Cargar dataset de evaluación

In [ ]:
import pandas as pd

# evaluacion_completa.csv -> columnas: Requirement, Answer, Source_chunks
# Source_chunks NO se usa: el RAG recupera desde cisco_ios_xe_16.11.1_chunks.jsonl.
raw = pd.read_csv(DRIVE_DATASET_PATH, encoding='utf-8')

for col in ['requirement', 'configuration']:
    assert col in raw.columns, 'Columna faltante: ' + col

df = pd.DataFrame({
    'requirement':  raw['requirement'].astype(str),
    'ground_truth': raw['configuration'].astype(str),
})
df['id'] = range(len(df))


def _count_config_lines(text):
    '''Cuenta lineas de configuracion reales del ground truth.
    Excluye prompts CLI y enable/configure terminal/end (igual criterio que normalize_config).
    '''
    n = 0
    for line in str(text).split(chr(10)):
        if '#' in line:
            line = line.split('#', 1)[-1]
        elif '>' in line:
            line = line.split('>', 1)[-1]
        line = ' '.join(line.split()).lower()
        if line and line not in ('configure terminal', 'end', 'enable', 'no_code'):
            n += 1
    return n


df['n_lines'] = df['ground_truth'].apply(_count_config_lines)

EVAL_SAMPLE_SIZE = None
EVAL_SAMPLE_SEED = 42

df['original_id'] = df['id']

if EVAL_SAMPLE_SIZE is not None and len(df) > EVAL_SAMPLE_SIZE:
    df = (
        df.sample(n=EVAL_SAMPLE_SIZE, random_state=EVAL_SAMPLE_SEED)
          .reset_index(drop=True)
    )
    df['id'] = range(len(df))

print('✅ Dataset cargado: ' + str(len(df)) + ' muestras')
print('   Columnas: ' + str(list(df.columns)))
print()
print('Distribucion n_lines:')
print(df['n_lines'].describe().to_string())
df.head(3)

✅ Dataset cargado: 150 muestras
   Columnas: ['requirement', 'ground_truth', 'id', 'n_lines', 'original_id']

Distribucion n_lines:
count    150.000000
mean       7.200000
std        6.006708
min        1.000000
25%        4.000000
50%        6.000000
75%        9.000000
max       50.000000


,requirement,ground_truth,id,n_lines,original_id
0,For DHCP On-Demand Address Pool Manager retrie...,R1# configure terminal\nR1(config)# ip dhcp po...,0,6,0
1,"For DHCP relay Option 82 retrieval, use the se...",R1# configure terminal\nR1(config)# ip dhcp-re...,1,7,1
2,Configure both ends of the R1-R2 point-to-poin...,R1# configure terminal\nR1(config)# interface ...,2,6,2


## 🗂️ Celda 6 — Cargar corpus RAG e índice FAISS

In [ ]:
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import faiss

# Reutilizar artefactos RAG si ya existen; si no, crearlos una sola vez.

def build_embedding_text(row):
    parts = [
        'Device type: ' + str(row.get('device_type', '')),
        'Product: '     + str(row.get('product', '')),
        'OS: '          + str(row.get('os', '')) + '  Version: ' + str(row.get('version', '')),
        'Guide: '       + str(row.get('configuration_guide', '')),
        'Chapter: '     + str(row.get('chapter', '')),
        'Section: '     + str(row.get('section', '')),
        '',
        'Commands:',
        str(row.get('commands', '')),
        '',
        'Examples:',
        str(row.get('examples', '')),
    ]
    return chr(10).join(parts).strip()

if USE_RAG:
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cpu')
else:
    embedding_model = None

if not USE_RAG:
    print('USE_RAG=False: se omite la carga de embeddings, metadata e indice FAISS.')
    emb_full = np.empty((0, 384), dtype='float32')
    meta_full = pd.DataFrame(columns=[
        'section_id', 'device_type', 'product', 'product_match_kind', 'os', 'version',
        'configuration_guide', 'chapter', 'section', 'commands', 'examples', 'rag_text'
    ])
elif os.path.exists(EMB_PATH) and os.path.exists(META_PATH):
    print('Artefactos RAG existentes encontrados. Cargando sin recrear...')
    emb_full  = np.load(EMB_PATH)
    meta_full = pd.read_parquet(META_PATH).reset_index(drop=True)
else:
    meta_full = pd.read_json(DRIVE_RAG_PATH, lines=True).reset_index(drop=True)
    meta_full['rag_text'] = meta_full.apply(build_embedding_text, axis=1)
    print('Vectorizando corpus completo: ' + str(len(meta_full)) + ' chunks')
    emb_full = embedding_model.encode(
        meta_full['rag_text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')
    np.save(EMB_PATH, emb_full)
    meta_full.to_parquet(META_PATH, index=False)

assert emb_full.shape[0] == len(meta_full), 'Desalineados: embeddings vs metadata'

print('OK Artefactos RAG cargados:')
print('   embeddings: ' + str(emb_full.shape))
print('   metadata  : ' + str(meta_full.shape))
print('   alineacion: fila N metadata == vector N embeddings')

BATFISH_TEXT_FIELDS = [
    'commands',
    'purpose',
    'examples',
    'section',
    'chapter',
    'configuration_guide',
    'block_type',
]

BATFISH_SHOW_FIELDS = [
    'commands',
]

BATFISH_EXCLUDE_TERMS = [
    'show ip route',
    'show bgp summary',
    'show interfaces',
    'show interface',
    'show arp',
    'show logging',
    'show processes',
    'show cpu',
    'show memory',
    'show platform',
    'show inventory',
    'show environment',
    'counter',
    'counters',
    'uptime',
    'traffic statistics',
    'interface statistics',
    'log messages',
    'syslog',
    'logging host',
    'snmp-server',
    'trap',
    'netflow',
    'flow monitor',
    'telemetry',
    'gnmi',
    'grpc',
    'ip sla',
    'event manager',
    'scheduler',
    'aaa ',
    'radius',
    'tacacs',
    'banner',
    'netconf',
    'restconf',
    'yang',
    'guest shell',
    'guestshell',
    'python script',
    'tcl script',
    'api',
    'shape average',
    'police ',
    'priority percent',
    'bandwidth percent',
    'queue',
    'jitter',
    'delay',
    'packet loss',
    'throughput',
    'domain lookup',
    'ntp',
    'ip http server',
    'ip http secure-server',
    'tftp',
    'ftp',
    'scp',
    'call-home',
    'license',
    'smart licensing',
    'dmvpn',
    'getvpn',
    'flexvpn',
    'pki',
    'certificate',
    'ikev2',
    'segment routing',
    'srv6',
    'pseudowire',
    'vpls',
    'l2vpn',
    'module',
    'transceiver',
    'poe',
    'stackwise',
    'fan',
    'temperature',
    'power supply',
    'asic',
    'atm',
    'pvc',
    'pvcs',
    'vpi',
    'vci',
    'aal5',
    'ancp',
    'inarp',
    'dsl',
    'pppoe',
    'pppoa',
    'broadband',
    'vpdn',
    'virtual-template',
    'dialer',
    'isdn',
    'frame-relay',
    'serial',
    'vxlan',
    'evpn',
    'nve',
    'mpls',
    'service instance',
    'service instances',
    'bridge-domain',
    'bridge domain',
    'ethernet cfm',
    'cfm',
    'oam',
    'evc',
    'ethernet service',
    'xconnect',
    'vfi',
    'qos',
    'class-map',
    'policy-map',
    'service-policy',
    'mqc',
    'fair-queue',
    'random-detect',
    'wred',
    'crypto',
    'ipsec',
    'ike',
    'trustpoint',
    'macsec',
    'zone-based',
    'zone security',
    'cts',
    'sgt',
    'dot1x',
    'mab',
    'device-tracking',
    'application hosting',
    'app-hosting',
    'iox',
    'appnav',
    'waas',
    'performance monitor',
    'avc',
    'umbrella',
    'mdns',
    'service-routing',
    'cns',
    'wireless',
    'wlan',
    'cellular',
    'lte',
    'mobile ipv6',
    'pmipv6',
    'debug',
    'traceroute',
    'monitor capture',
    'monitoring',
    'verifying',
    'troubleshooting',
]

BATFISH_RUNTIME_ONLY_TERMS = [
    'show spanning-tree',
    'show mac address-table',
    'show etherchannel',
    'show lacp',
    'show pagp',
    'show ip mroute',
    'show ip pim',
    'show ip igmp',
    'cam table',
    'suspended',
]


def norm(s):
    return ' '.join(str(s).strip().lower().split())


def _as_list(value):
    if isinstance(value, (list, tuple, set)):
        return [str(v) for v in value]
    if value is None:
        return []
    if isinstance(value, str) and ',' in value:
        return [part.strip() for part in value.split(',') if part.strip()]
    return [str(value)]


def literal_or_match(chunk_value, allowed_values):
    chunk_text = norm(chunk_value)
    return any(
        norm(allowed) and norm(allowed) in chunk_text
        for allowed in _as_list(allowed_values)
    )


def chunk_matches_arch(row, arch):
    return (
        literal_or_match(row.get('os', ''), arch['os']) and
        literal_or_match(row.get('version', ''), arch['version']) and
        literal_or_match(row.get('device_type', ''), arch['device_type']) and
        (
            not USE_PRODUCT_FILTER or
            literal_or_match(row.get('product', ''), arch['product'])
        )
    )


def chunk_text(row, fields=BATFISH_TEXT_FIELDS):
    return ' '.join(norm(row.get(field, '')) for field in fields)


def has_disallowed_show(text):
    if 'show ' not in text:
        return False
    return 'show running-config' not in text


def term_in_text(term, text):
    term = norm(term)
    if not term:
        return False
    pattern = r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])'
    return re.search(pattern, text) is not None


def batfish_compatible(row):
    show_text = chunk_text(row, fields=BATFISH_SHOW_FIELDS)
    if has_disallowed_show(show_text):
        return False

    full_text = chunk_text(row, fields=BATFISH_TEXT_FIELDS)
    if any(term_in_text(term, full_text) for term in BATFISH_EXCLUDE_TERMS):
        return False
    if any(term_in_text(term, full_text) for term in BATFISH_RUNTIME_ONLY_TERMS):
        return False
    return True


def build_rag_text(row):
    parts = [
        'Device type: ' + str(row.get('device_type', '')),
        'Product: '     + str(row.get('product', '')),
        'OS: '          + str(row.get('os', '')) + '  Version: ' + str(row.get('version', '')),
        'Guide: '       + str(row.get('configuration_guide', '')),
        'Chapter: '     + str(row.get('chapter', '')),
        'Section: '     + str(row.get('section', '')),
        '',
        'Commands:',
        str(row.get('commands', '')),
        '',
        'Examples:',
        str(row.get('examples', '')),
    ]
    return chr(10).join(parts).strip()


# El filtro arquitectonico se aplica chunk por chunk con match literal.
mask_arch = meta_full.apply(lambda row: chunk_matches_arch(row, ARCH_BASE), axis=1).astype(bool)
rag_df = meta_full[mask_arch]
chunks_after_arch = int(len(rag_df))
print('Chunks RAG iniciales: ' + str(len(meta_full)))
print('Chunks despues de filtro ARCH_BASE: ' + str(chunks_after_arch))

if USE_RAG and USE_BATFISH_COMPAT_FILTER:
    mask_batfish = rag_df.apply(batfish_compatible, axis=1).astype(bool)
    rag_df = rag_df[mask_batfish]
chunks_after_batfish = int(len(rag_df))
print('Chunks despues de filtro BATFISH_EXCLUDE_TERMS: ' + str(chunks_after_batfish))

selected_pos = rag_df.index.to_numpy()
rag_df = rag_df.reset_index(drop=True)
if 'rag_text' not in rag_df.columns:
    rag_df['rag_text'] = rag_df.apply(build_rag_text, axis=1)

FILTERED_RAG_JSONL_PATH = DRIVE_RESULTS_FOLDER + '/rag_filtered_arch_batfish.jsonl'
rag_df.to_json(FILTERED_RAG_JSONL_PATH, orient='records', lines=True, force_ascii=False)
print('JSONL RAG filtrado final: ' + FILTERED_RAG_JSONL_PATH)

rag_embeddings = emb_full[selected_pos].astype('float32')
assert len(rag_embeddings) == len(rag_df), 'slice desalineado: vectores vs metadata filtrada'
if USE_RAG and len(rag_df) == 0:
    raise ValueError('El filtro arquitectonico/Batfish vacio el corpus. Revisa ARCH_BASE.')

# Indice FAISS unico construido sobre el subconjunto filtrado.
if USE_RAG:
    rag_index = faiss.IndexFlatIP(rag_embeddings.shape[1])
    rag_index.add(rag_embeddings)
else:
    rag_index = None

# Reutilizar el mismo modelo para embeber las queries.
if USE_RAG:
    print('Modelo de embeddings listo para queries: ' + EMBEDDING_MODEL_NAME)
else:
    print('Modelo de embeddings no cargado porque USE_RAG=False')

print(chr(10) + 'OK Subconjunto filtrado: ' + str(len(rag_df)) + ' chunks')
print('   vectores slice: ' + str(rag_embeddings.shape))
print('   indice FAISS filtrado: ' + str(rag_index.ntotal if rag_index is not None else 0) + ' vectores')
print('   corpus completo pre-embebido: ' + str(len(meta_full)) + ' chunks')

print(chr(10) + 'chunks por device_type:')
ARCH_DEVICE_TYPE_COUNTS = rag_df['device_type'].value_counts(dropna=False).to_dict()
print(rag_df['device_type'].value_counts(dropna=False).to_string())

print(chr(10) + 'chunks por product:')
ARCH_PRODUCT_COUNTS = rag_df['product'].value_counts(dropna=False).to_dict()
print(rag_df['product'].value_counts(dropna=False).head(30).to_string())


def _device_scope_mask(df_in, scope):
    return df_in['device_type'].apply(lambda x: literal_or_match(x, [scope]))


RAG_DEVICE_SCOPES = ['router', 'switch']
rag_device_indexes = {}
rag_device_positions = {}
RAG_DEVICE_RETRIEVAL_COUNTS = {}

for scope in RAG_DEVICE_SCOPES:
    local_positions = np.flatnonzero(_device_scope_mask(rag_df, scope).to_numpy())
    rag_device_positions[scope] = local_positions
    RAG_DEVICE_RETRIEVAL_COUNTS[scope] = int(len(local_positions))
    if len(local_positions) > 0:
        local_index = faiss.IndexFlatIP(rag_embeddings.shape[1])
        local_index.add(rag_embeddings[local_positions])
        rag_device_indexes[scope] = local_index
    else:
        rag_device_indexes[scope] = None

print(chr(10) + 'chunks disponibles para retrieval por scope:')
for scope in RAG_DEVICE_SCOPES:
    print('   ' + scope + ': ' + str(RAG_DEVICE_RETRIEVAL_COUNTS[scope]))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

## 📝 Celda 7 — Topología y prompts (enfoque RAG v3)

In [ ]:
# -- Topologia de red (igual que topo_v1) ---------------------------------
NETWORK_CONTEXT = '''=== NETWORK TOPOLOGY ===

Interface table:
DEVICE  INTERFACE    IP ADDRESS    MASK             NEIGHBOR  NEIGHBOR IFACE
R1      Ethernet0/0  10.0.1.1      255.255.255.0    SW1       Fa0/24
R1      Ethernet0/1  10.0.12.1     255.255.255.0    R2        Ethernet0/0
R1      Ethernet0/2  10.0.14.1     255.255.255.0    R4        Ethernet0/0
R2      Ethernet0/0  10.0.12.2     255.255.255.0    R1        Ethernet0/1
R2      Ethernet0/1  10.0.23.1     255.255.255.0    R3        Ethernet0/0
R2      Ethernet0/2  10.0.24.1     255.255.255.0    R4        Ethernet0/1
R3      Ethernet0/0  10.0.23.2     255.255.255.0    R2        Ethernet0/1
R3      Ethernet0/1  10.0.34.1     255.255.255.0    R4        Ethernet0/2
R3      Ethernet0/2  10.0.2.1      255.255.255.0    SW2       Fa0/24
R4      Ethernet0/0  10.0.14.2     255.255.255.0    R1        Ethernet0/2
R4      Ethernet0/1  10.0.24.2     255.255.255.0    R2        Ethernet0/2
R4      Ethernet0/2  10.0.34.2     255.255.255.0    R3        Ethernet0/1

Hosts:
- h1: 10.0.1.10/24  gw 10.0.1.1   -> SW1 Fa0/1  (VLAN 10)
- h2: 10.0.2.10/24  gw 10.0.2.1   -> SW2 Fa0/1  (VLAN 30)

Switch ports:
- SW1 Fa0/1  : access VLAN 10, connected to h1
- SW1 Fa0/24 : trunk VLANs 10,20, connected to R1 Ethernet0/0
- SW2 Fa0/1  : access VLAN 30, connected to h2
- SW2 Fa0/24 : trunk VLANs 30,40, connected to R3 Ethernet0/2
'''

# -- Prompts de generacion: NoRAG y RAG ------------------------------------
GENERATION_PROMPT_NO_RAG = '''You are a Cisco network engineer. Generate the exact CLI configuration commands to fulfill the requirement using the network topology.

RULES:
- Output ONLY configuration commands with CLI mode prompts.
- Do not output markdown fences such as ``` or ```plaintext.
- Do not output explanations, comments, bullets, labels, or extra text.
- Use the device name from the requirement as the CLI hostname (for example, R1# and R1(config)#; SW1# and SW1(config)#).
- Never use generic documentation prompts or hostnames such as Device>, Device#, Router>, Router#, Switch>, Switch#, Router(config)#, or Switch(config)# when a topology device is available.
- Include correct sub-mode prompts: (config-if)#, (config-router)#, (config-line)#, (config-vlan)#, (config-router-af)#, (config-vrf)#, (config-ext-nacl)#, etc.
- Begin each device block with <DEVICE># configure terminal and close the block with end.
- Use only device names, interfaces, IP addresses, VLANs, ACL names, route targets, next hops, timers, usernames, and passwords stated in the requirement or topology.

Network topology:
{network_context}

Requirement:
{requirement}

Configuration:'''

GENERATION_PROMPT_RAG1 = '''You are a Cisco network engineer. Generate the exact CLI configuration commands to fulfill the requirement using the network topology and the retrieved documentation.

RULES:
- Output ONLY configuration commands with CLI mode prompts.
- Do not output markdown fences such as ``` or ```plaintext.
- Do not output explanations, comments, bullets, labels, or extra text.
- Use the retrieved documentation only for command syntax and configuration sequence.
- Do not copy example device names, prompts, IP addresses, interfaces, VLANs, ACL names, usernames, passwords, or arbitrary values from the retrieved documentation.
- Use exactly the device names, interfaces, IP addresses, VLANs, ACL names, route targets, next hops, timers, usernames, and passwords stated in the requirement or topology.
- Use the device name from the requirement as the CLI hostname (for example, R1# and R1(config)#; SW1# and SW1(config)#).
- Never output generic documentation prompts or hostnames such as Device>, Device#, Router>, Router#, Switch>, Switch#, Router(config)#, or Switch(config)#.
- If a documentation example conflicts with the requirement or topology, follow the requirement and topology.
- Prefer classic, widely-supported IOS syntax over newer address-family variants unless the requirement explicitly asks for it. For OSPF, use OSPFv2 (router ospf <pid> with network <addr> <wildcard> area <id>) unless the requirement explicitly requires OSPFv3 or IPv6. The retrieved documentation may show newer variants; do NOT copy a variant the requirement did not ask for.
- Include correct sub-mode prompts: (config-if)#, (config-router)#, (config-line)#, (config-vlan)#, (config-router-af)#, (config-vrf)#, (config-ext-nacl)#, etc.
- Begin each device block with <DEVICE># configure terminal and close the block with end.

Network topology:
{network_context}

Retrieved Cisco documentation:
{retrieved_context}

Requirement:
{requirement}

Configuration:'''

GENERATION_PROMPT_RAG = '''You are a Cisco network engineer. Generate the exact CLI configuration commands to fulfill the requirement using the network topology and the retrieved documentation.

RULES:
- Output ONLY configuration commands with CLI mode prompts.
- Do not output markdown fences such as ``` or ```plaintext.
- Do not output explanations, comments, bullets, labels, or extra text.
- Use the retrieved documentation only for command syntax and configuration sequence.
- Do not copy example device names, prompts, IP addresses, interfaces, VLANs, ACL names, usernames, passwords, or arbitrary values from the retrieved documentation.
- Use exactly the device names, interfaces, IP addresses, VLANs, ACL names, route targets, next hops, timers, usernames, and passwords stated in the requirement or topology.
- Use the device name from the requirement as the CLI hostname (for example, R1# and R1(config)#; SW1# and SW1(config)#).
- Never output generic documentation prompts or hostnames such as Device>, Device#, Router>, Router#, Switch>, Switch#, Router(config)#, or Switch(config)#.
- If a documentation example conflicts with the requirement or topology, follow the requirement and topology.
- Prefer classic, widely-supported IOS syntax over newer variants unless the requirement explicitly asks for a protocol version, IPv6, address-family mode, named mode, redistribution, route-reflector, VRF-aware variant, policy feature, or other advanced variant.
- If the retrieved documentation describes an advanced or different variant that the requirement did not ask for, ignore that chunk and generate the classic configuration from the requirement and topology.
- For OSPF, use OSPFv2 (router ospf <pid> with network <addr> <wildcard> area <id>) unless the requirement explicitly requires OSPFv3 or IPv6.
- For EIGRP, use classic autonomous-system syntax (router eigrp <asn>, network <network>, no auto-summary when appropriate) unless the requirement explicitly asks for named EIGRP or address-family mode.
- For BGP, use classic IPv4 unicast peering syntax (router bgp <asn>, neighbor <ip> remote-as <asn>, network <addr> mask <mask>) unless the requirement explicitly asks for VRF, address-family, route-map, policy, prefix-list, aggregation, or another advanced BGP feature.
- For NAT, match the requested NAT variant exactly: PAT/overload uses ip nat inside source list ... interface ... overload; dynamic NAT uses an ACL plus ip nat pool; static NAT uses ip nat inside source static. Do not copy outside-source, CGN, ALG, VRF-aware, or route-map NAT unless explicitly requested.
- For ACLs, when the requirement asks to allow, deny, block, permit, restrict, or filter traffic, create the access list and apply it to the correct interface/direction when the requirement implies or states application.
- For DHCP, use classic ip dhcp excluded-address, ip dhcp pool, network, default-router, dns-server, and ip helper-address syntax unless the requirement explicitly asks for an advanced DHCP feature.
- For static routes, use classic ip route <destination> <mask> <next-hop|interface> syntax unless the requirement explicitly asks for VRF, tracking, route-map, recursive, or another advanced static-route feature.
- Do not add an IP address, VLAN, subinterface, routing process, ACL body, NAT rule, or feature command only because it appears in a chunk; add it only when required by the requirement or topology.
- Include correct sub-mode prompts: (config-if)#, (config-router)#, (config-line)#, (config-vlan)#, (config-router-af)#, (config-vrf)#, (config-ext-nacl)#, etc.
- Begin each device block with <DEVICE># configure terminal and close the block with end.

Network topology:
{network_context}

Retrieved Cisco documentation:
{retrieved_context}

Requirement:
{requirement}

Configuration:'''

# -- Prompt de normalizacion de intent: salida texto simple ----------------
INTENT_NORMALIZATION_PROMPT = '''You convert a Cisco network configuration requirement into an enriched retrieval query for a documentation RAG corpus.

Goal: improve retrieval of useful Cisco configuration documentation while preserving the original requirement unchanged.

Return exactly one plain text line in one of these formats:
<original requirement unchanged> || <retrieval enrichment terms>
<original requirement unchanged>

The enrichment prefix should contain concise Cisco documentation/search terms that describe the requested feature, command family, and classic configuration variant. Use general Cisco terms only; do not hardcode dataset examples, topology values, chunk IDs, product names, OS versions, or device-specific values.

Good enrichment terms are generic documentation concepts such as:
- interface configuration, ip address, no shutdown, shutdown, description, bandwidth, mtu, speed, duplex
- creating named extended access list, standard access list, ip access-list extended, permit, deny, protocol, source, destination, wildcard, host, any, eq, log, applying access list to interface, ip access-group, access-class
- configuring static routes, ip route, default route, floating static route, next-hop, administrative distance
- enabling OSPF, router ospf, network wildcard-mask area, passive-interface, router-id, ip ospf cost, ip ospf priority
- enabling EIGRP autonomous system, router eigrp, network, passive-interface, no auto-summary
- configuring BGP peer, router bgp, neighbor remote-as, network mask, eBGP, iBGP
- NAT overload, PAT, ip nat inside, ip nat outside, ip nat inside source list interface overload, dynamic NAT, ip nat pool, static NAT, ip nat inside source static
- configuring DHCP address pool, ip dhcp excluded-address, ip dhcp pool, network, default-router, dns-server, lease, DHCP relay, ip helper-address
- VRF definition, rd, route-target, address-family ipv4, vrf forwarding
- line vty, login local, transport input ssh, username, enable secret, crypto key generate rsa

Feature selection guidance:
- For allow, permit, deny, block, restrict, filter, traffic direction, application ports, HTTP, HTTPS, FTP, TFTP, Telnet, SSH, SMTP, SNMP, DNS, ICMP, TCP, UDP, or IP traffic, enrich for ACL creation and ACL application.
- For NAT, PAT, overload, inside/outside, translation, Internet access, or address sharing, enrich with the specific NAT variant if present: overload/PAT, dynamic NAT, or static NAT.
- For OSPF without explicit IPv6 or OSPFv3, enrich for classic OSPFv2 enabling OSPF, router ospf, and network wildcard-mask area. Do not add OSPFv3/address-family terms unless explicitly requested.
- For EIGRP without explicit named/address-family wording, enrich for classic autonomous-system EIGRP. Do not add named EIGRP or address-family terms unless explicitly requested.
- For BGP without explicit policy/VRF/address-family wording, enrich for basic BGP peer and network advertisement. Do not add route-map, prefix-list, VRF, aggregation, or address-family terms unless explicitly requested.
- For DHCP relay, include ip helper-address. For DHCP server/pool, include DHCP address pool terms.
- For static route, default route, floating route, backup route, next-hop, or administrative distance, enrich for configuring static routes.
- For interface enable/disable only, enrich for interface configuration and no shutdown/shutdown only; do not add ip address unless the requirement asks for addressing.
- If multiple feature families are required, include compact terms for each family.
- If no feature family is clear, return the original requirement unchanged with no separator.

Strict rules:
- Preserve the original requirement exactly after the separator. Do not paraphrase it.
- The prefix must not contain device names, host names, exact IP addresses, exact interface names, ACL names, usernames, passwords, route targets, next-hops, AS numbers, VLAN IDs, timers, or topology-specific values.
- The prefix must not contain OS versions, product names, architecture names, dataset-specific wording, chunk IDs, labels, explanations, JSON, bullets, markdown, or CLI configuration output.
- The prefix must not contain invented examples or placeholders such as Router1, Ethernet0/0, 10.0.0.0, ACL_NAME, AS_NUMBER, VLAN_ID, source-ip, destination-ip, or <value>.
- Do not start with meta words such as Normalize, Query, Intent, Result, Output, or Find documentation.
- Return one plain text line only.

User requirement:
__REQUIREMENT__
'''

print('OK Topologia y prompts definidos (RAG simple con filtro arquitectonico)')

## 🤖 Celda 8 — Cargar modelo (Qwen2.5-7B-Instruct 4bit + LoRA opcional)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Cargando ' + BASE_MODEL_PATH + ' (4bit)...')
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, token=HF_TOKEN)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=quant_config,
    device_map='auto',
    token=HF_TOKEN,
)

if USE_FINETUNED:
    print('Aplicando LoRA desde ' + ADAPTER_PATH + '...')
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, token=HF_TOKEN)
else:
    model = base_model

model.eval()
vram_used = torch.cuda.memory_allocated(0) / 1024**3
print(chr(10) + '✅ ' + MODEL_NAME + ' listo  |  VRAM: ' + '{:.2f}'.format(vram_used) + ' GB')

## ⚙️ Celda 9 — Funciones de inferencia y RAG (enfoque v3)

In [ ]:
def build_prompt(messages, plain_prompt):
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    return plain_prompt


def clean_normalized_intent(text):
    '''Limpia fences y prefijos; la salida esperada es texto plano.'''
    text = str(text).strip()
    text = text.replace('```json', '').replace('```', '').strip()
    for prefix in ['Normalized intent:', 'Intent:', 'Output:', 'Result:', 'Query:', 'Normalize:', 'Find documentation:']:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
    if text.lower().startswith('normalize '):
        text = text[len('normalize '):].strip()
    if text.lower().startswith('find documentation for '):
        text = text[len('find documentation for '):].strip()
    return text


def normalize_intent(requirement, max_new_tokens=MAX_NEW_TOKENS_INTENT):
    '''Normaliza el requerimiento en una unica query tecnica para retrieval.'''
    try:
        plain    = INTENT_NORMALIZATION_PROMPT.replace('__REQUIREMENT__', requirement)
        messages = [{'role': 'user', 'content': plain}]
        prompt   = build_prompt(messages, plain)
        inputs   = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=2048
        ).to('cuda:0')
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
                temperature=None, top_p=None, top_k=None)
        text = tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        text = clean_normalized_intent(text)
        return text if text else requirement
    except Exception as e:
        print('  normalize_intent warning: ' + str(e))
        return requirement


def build_rag_query(requirement, normalized_intent):
    '''Query al RAG: requerimiento original o normalizacion opcional sin etiquetas meta.'''
    requirement = str(requirement).strip()
    normalized_intent = str(normalized_intent).strip()
    if USE_INTENT_NORMALIZATION and normalized_intent:
        return (normalized_intent + chr(10) + requirement).strip()
    return requirement


def infer_requirement_device_scopes(requirement):
    '''Detecta si la consulta debe buscar en router, switch o ambos.'''
    text = ' ' + str(requirement).lower() + ' '
    router_patterns = [
        ' r1', ' r2', ' r3', ' r4',
        'router1', 'router2', 'router3', 'router4',
        ' router ', ' routers ',
    ]
    switch_patterns = [
        ' sw1', ' sw2',
        'switch1', 'switch2',
        ' switch ', ' switches ',
    ]
    has_router = any(p in text for p in router_patterns)
    has_switch = any(p in text for p in switch_patterns)
    if has_router and has_switch:
        return ['router', 'switch']
    if has_router:
        return ['router']
    if has_switch:
        return ['switch']
    return ['router', 'switch']


def _row_to_chunk(row, score, retrieval_scope):
    return {
        'score':               float(score),
        'section_id':          str(row.get('section_id', '')),
        'device_type':         str(row.get('device_type', '')),
        'product':             str(row.get('product', '')),
        'product_match_kind':  str(row.get('product_match_kind', '')),
        'os':                  str(row.get('os', '')),
        'version':             str(row.get('version', '')),
        'configuration_guide': str(row.get('configuration_guide', '')),
        'chapter':             str(row.get('chapter', '')),
        'section':             str(row.get('section', '')),
        'commands':            str(row.get('commands', '')),
        'examples':            str(row.get('examples', '')),
        'retrieval_scope':     retrieval_scope,
    }


def _search_scope(q_emb, scope, top_k):
    '''Busca dentro del indice FAISS del scope indicado.'''
    local_index = rag_device_indexes.get(scope)
    local_positions = rag_device_positions.get(scope, np.array([], dtype=int))
    if local_index is None or len(local_positions) == 0:
        return []
    scores, local_indices = local_index.search(q_emb, min(top_k, local_index.ntotal))
    results = []
    for score, local_idx in zip(scores[0], local_indices[0]):
        if local_idx < 0:
            continue
        if USE_RAG_MIN_SCORE and score < RAG_MIN_SCORE:
            continue
        rag_pos = int(local_positions[int(local_idx)])
        row = rag_df.iloc[rag_pos]
        results.append(_row_to_chunk(row, score, scope))
    return results


def retrieve_rag_chunks(query, requirement, top_k=3):
    '''Recupera top_k por tipo detectado, evitando mezclar router/switch innecesariamente.'''
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype('float32')
    scopes = infer_requirement_device_scopes(requirement)
    results = []
    seen_sections = set()

    for scope in scopes:
        scoped_results = _search_scope(q_emb, scope, top_k)
        for chunk in scoped_results:
            key = chunk['section_id'] or (
                chunk['configuration_guide'], chunk['chapter'], chunk['section'], chunk['retrieval_scope'])
            if key in seen_sections:
                continue
            results.append(chunk)
            seen_sections.add(key)

    if not results:
        scores, indices = rag_index.search(q_emb, min(top_k, rag_index.ntotal))
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            if USE_RAG_MIN_SCORE and score < RAG_MIN_SCORE:
                continue
            row = rag_df.iloc[int(idx)]
            results.append(_row_to_chunk(row, score, 'architecture_fallback'))

    return results


def build_retrieved_context(chunks):
    '''Formato verbose: [DOCUMENTATION CHUNK i] con Score, OS, Version, etc.'''
    if not chunks:
        return ''
    blocks = []
    for i, c in enumerate(chunks, 1):
        parts = [
            '[DOCUMENTATION CHUNK ' + str(i) + ']',
            'Score: '       + '{:.4f}'.format(c['score']),
            'Retrieval scope: ' + c.get('retrieval_scope', ''),
            'Device type: ' + c['device_type'],
            'Product: '     + c['product'],
            'OS: '          + c['os'],
            'Version: '     + c['version'],
            'Guide: '       + c['configuration_guide'],
            'Chapter: '     + c['chapter'],
            'Section: '     + c['section'],
            '',
            'Commands:',
            c['commands'],
            '',
            'Examples:',
            c['examples'],
        ]
        blocks.append(chr(10).join(parts).strip())
    return (chr(10) + chr(10)).join(blocks)


def generate_config(requirement, retrieved_context, max_new_tokens=MAX_NEW_TOKENS):
    '''Selecciona prompt NoRAG o RAG segun exista contexto recuperado.'''
    try:
        prompt_template = GENERATION_PROMPT_RAG if str(retrieved_context).strip() else GENERATION_PROMPT_NO_RAG
        plain = prompt_template.format(
            network_context=NETWORK_CONTEXT,
            requirement=requirement,
            retrieved_context=retrieved_context)
        messages = [{'role': 'user', 'content': plain}]
        prompt   = build_prompt(messages, plain)
        inputs   = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=4096
        ).to('cuda:0')
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
                temperature=None, top_p=None, top_k=None)
        return tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        print('  generate_config error: ' + str(e))
        return 'ERROR'


print('OK Funciones RAG e inferencia definidas (retrieval separado por device_type)')

## 📊 Celda 10 — Funciones de métricas (ROUGE normalizado + BERTScore)

In [ ]:
import numpy as np


def normalize_config(text):
    '''Elimina prompts CLI, configure terminal y end para comparacion justa.
    El prompt v3 genera configure terminal/end; se eliminan aqui para que
    el ROUGE sea comparable con el ground truth de topo_v1 que no los tiene.
    '''
    lines = []
    for line in text.split(chr(10)):
        if '#' in line:
            line = line.split('#', 1)[-1]
        elif '>' in line:
            line = line.split('>', 1)[-1]
        line = ' '.join(line.split()).lower()
        if line and line not in ('configure terminal', 'end', 'enable', 'no_code'):
            lines.append(line)
    return lines


def compute_rouge(predictions, references):
    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not ref or pred == 'ERROR':
            continue
        pred_norm = ' '.join(normalize_config(pred))
        ref_norm  = ' '.join(normalize_config(ref))
        if not pred_norm or not ref_norm:
            continue
        s = scorer.score(ref_norm, pred_norm)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)

    def safe(lst):    return round(float(np.mean(lst)), 4) if lst else 0.0
    def safestd(lst): return round(float(np.std(lst)),  4) if lst else 0.0

    return {
        'rouge1': safe(r1), 'rouge1_std': safestd(r1),
        'rouge2': safe(r2), 'rouge2_std': safestd(r2),
        'rougeL': safe(rl), 'rougeL_std': safestd(rl),
    }


def compute_bertscore(predictions, references):
    from bert_score import score as bscore
    valid = [(p, r) for p, r in zip(predictions, references)
             if p and r and p != 'ERROR']
    if not valid:
        return {'bertscore_p': 0.0, 'bertscore_r': 0.0,
                'bertscore_f1': 0.0, 'bertscore_f1_std': 0.0}
    preds, refs = zip(*valid)
    for model_name, num_layers in [('microsoft/codebert-base', 12), ('roberta-large', None)]:
        try:
            kwargs = {'lang': 'en', 'model_type': model_name,
                      'verbose': False, 'batch_size': 16}
            if num_layers:
                kwargs['num_layers'] = num_layers
            P, R, F1 = bscore(list(preds), list(refs), **kwargs)
            print('    BERTScore model: ' + model_name)
            return {
                'bertscore_p':      round(float(P.mean()),  4),
                'bertscore_r':      round(float(R.mean()),  4),
                'bertscore_f1':     round(float(F1.mean()), 4),
                'bertscore_f1_std': round(float(F1.std()),  4),
            }
        except Exception as e:
            print('    ⚠️ ' + model_name + ': ' + str(e))
    return {'bertscore_p': 0.0, 'bertscore_r': 0.0,
            'bertscore_f1': 0.0, 'bertscore_f1_std': 0.0}


print('✅ Metricas definidas (ROUGE normalizado + BERTScore)')

## 🚀 Celda 11 — Evaluación principal
> ⏱️ Ligeramente mas lento que v1 por normalización JSON (180 tokens) y MAX_NEW_TOKENS=512. 140 muestras ≈ 20-25 min.

In [ ]:
import time
from datetime import datetime

SEP = '=' * 65
print(SEP)
print('EXPERIMENTO : ' + MODEL_NAME)
print('USE_RAG     : ' + str(USE_RAG))
print('Muestras    : ' + str(len(df)))
print('Inicio      : ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print(SEP)

predictions        = []
latencies          = []
retrieved_logs     = []
normalized_intents = []

for i, (_, row) in enumerate(df.iterrows()):
    if i % 20 == 0:
        vram = torch.cuda.memory_allocated(0) / 1024**3
        print('  [' + str(i + 1).rjust(3) + '/' + str(len(df)) + ']'
              '  VRAM: ' + '{:.2f}'.format(vram) + ' GB')

    requirement = row['requirement']

    t0 = time.time()

    if USE_RAG:
        normalized_intent = normalize_intent(requirement) if USE_INTENT_NORMALIZATION else ''
        rag_query         = build_rag_query(requirement, normalized_intent)
        retrieval_scopes  = infer_requirement_device_scopes(requirement)
        chunks            = retrieve_rag_chunks(rag_query, requirement, top_k=RAG_TOP_K)
        retrieved_context = build_retrieved_context(chunks)
    else:
        normalized_intent = ''
        rag_query         = ''
        chunks            = []
        retrieval_scopes  = []
        retrieved_context = ''

    pred = generate_config(requirement, retrieved_context)

    latencies.append(time.time() - t0)
    predictions.append(pred)
    normalized_intents.append(normalized_intent)

    retrieved_logs.append({
        'id':                 int(row['id']),
        'requirement':        requirement,
        'n_lines':            int(row['n_lines']),
        'normalized_intent':  normalized_intent,
        'rag_query':          rag_query,
        'retrieval_scopes':   retrieval_scopes,
        'chunks_retrieved':   [c['section_id'] for c in chunks],
        'chunk_device_types': [c['device_type'] for c in chunks],
        'chunk_products':     [c['product'] for c in chunks],
        'chunk_versions':     [c['version'] for c in chunks],
        'chunk_retrieval_scopes': [c.get('retrieval_scope', '') for c in chunks],
        'top_score':          round(chunks[0]['score'], 4) if chunks else None,
    })

references  = df['ground_truth'].tolist()
error_count = predictions.count('ERROR')
nocode_count = sum(1 for p in predictions if isinstance(p, str) and p.strip().upper() == 'NO_CODE')

total_min = sum(latencies) / 60
print(chr(10) + 'OK Inferencia completada - ' + '{:.1f}'.format(total_min) + ' min totales')
print('   Errores: ' + str(error_count) + '  NO_CODE: ' + str(nocode_count))

print(chr(10) + 'Calculando ROUGE (normalizado)...')
rouge_metrics = compute_rouge(predictions, references)

print('Calculando BERTScore...')
bert_metrics  = compute_bertscore(predictions, references)

avg_time = float(np.mean(latencies))

SEP2 = '=' * 55
print(chr(10) + SEP2)
print('  RESULTADOS GLOBALES - ' + MODEL_NAME)
print(SEP2)
print('  ROUGE-1 : ' + '{:.4f}'.format(rouge_metrics['rouge1']) +
      '  (+-' + '{:.4f}'.format(rouge_metrics['rouge1_std']) + ')')
print('  ROUGE-2 : ' + '{:.4f}'.format(rouge_metrics['rouge2']) +
      '  (+-' + '{:.4f}'.format(rouge_metrics['rouge2_std']) + ')')
print('  ROUGE-L : ' + '{:.4f}'.format(rouge_metrics['rougeL']) +
      '  (+-' + '{:.4f}'.format(rouge_metrics['rougeL_std']) + ')')
print('  BERTScore-F1: ' + '{:.4f}'.format(bert_metrics['bertscore_f1']) +
      '  (+-' + '{:.4f}'.format(bert_metrics['bertscore_f1_std']) + ')')
print('  Tiempo/muestra: ' + '{:.2f}'.format(avg_time) + 's  |  '
      'Errores: ' + str(error_count) + '  NO_CODE: ' + str(nocode_count))
print(SEP2)


## 📈 Celda 12 — Desglose por complejidad
> `evaluacion_completa.csv` no trae `rag_type` ni `feature_group`, así que solo se reporta el desglose por complejidad (n_lines del ground truth).

In [ ]:
SEP = '=' * 65

# evaluacion_completa.csv no incluye rag_type ni feature_group:
# esos desgloses no aplican y quedan vacios.
rag_type_results = {}
fg_results       = {}

print(SEP)
print('  DESGLOSE POR complejidad (n_lines) — ' + MODEL_NAME)
print(SEP)
print('bucket'.ljust(15) + '  ' + 'n'.rjust(4) + '  ' +
      'ROUGE-1'.rjust(8) + '  ' + 'ROUGE-L'.rjust(8))
print('-' * 45)

complexity_results = {}
for label, mask in [
    ('corta  (<=3)', df['n_lines'].astype(int) <= 3),
    ('media  (4-7)', (df['n_lines'].astype(int) >= 4) & (df['n_lines'].astype(int) <= 7)),
    ('larga  (>7) ', df['n_lines'].astype(int) > 7),
]:
    idx     = df.index[mask].tolist()
    preds_s = [predictions[i] for i in idx]
    refs_s  = [references[i]  for i in idx]
    r = compute_rouge(preds_s, refs_s)
    complexity_results[label] = {**r, 'n': len(idx)}
    print(label.ljust(15) + '  ' + str(len(idx)).rjust(4) + '  ' +
          '{:.4f}'.format(r['rouge1']).rjust(8) + '  ' +
          '{:.4f}'.format(r['rougeL']).rjust(8))

## 💾 Celda 13 — Guardar resultados en Google Drive

In [ ]:
from datetime import datetime
import json

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
filename    = 'results_' + MODEL_NAME + '_' + timestamp + '.json'
output_path = os.path.join(DRIVE_RESULTS_FOLDER, filename)

output = {
    'timestamp':             timestamp,
    'model':                 MODEL_NAME,
    'rag_approach':          'v7_device_split_arch_prefilter_top3',
    'arch_base':             ARCH_BASE,
    'use_product_filter':    USE_PRODUCT_FILTER,
    'use_batfish_compat_filter': USE_BATFISH_COMPAT_FILTER,
    'strict_product':        STRICT_PRODUCT,
    'rag_subset_size':       int(len(rag_df)),
    'rag_device_type_counts': ARCH_DEVICE_TYPE_COUNTS,
    'rag_product_counts':    ARCH_PRODUCT_COUNTS,
    'rag_device_retrieval_counts': RAG_DEVICE_RETRIEVAL_COUNTS,
    'use_finetuned':         USE_FINETUNED,
    'use_rag':               USE_RAG,
    'use_intent_normalization': USE_INTENT_NORMALIZATION,
    'adapter_path':          ADAPTER_PATH if USE_FINETUNED else None,
    'dataset':               DRIVE_DATASET_PATH,
    'rag_embeddings':        EMB_PATH,
    'rag_metadata':          META_PATH,
    'rag_top_k':             RAG_TOP_K,
    'use_rag_min_score':     USE_RAG_MIN_SCORE,
    'rag_min_score':         RAG_MIN_SCORE,
    'max_new_tokens':        MAX_NEW_TOKENS,
    'max_new_tokens_intent': MAX_NEW_TOKENS_INTENT,
    'embedding_model':       EMBEDDING_MODEL_NAME,
    'total_samples':         len(df),
    'error_count':           int(error_count),
    'nocode_count':          int(nocode_count),
    'total_time_s':          round(sum(latencies), 2),
    'avg_time_s':            round(avg_time, 3),
    **rouge_metrics,
    **bert_metrics,
    'by_rag_type':           rag_type_results,
    'by_feature_group':      fg_results,
    'by_complexity':         complexity_results,
    'predictions':           predictions,
    'references':            references,
    'normalized_intents':    normalized_intents,
    'retrieved_logs':        retrieved_logs,
}

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print('OK Resultados guardados:')
print('   ' + output_path)


## 🔍 Celda 14 — Inspección de predicciones
ROUGE-1 normalizado (configure terminal/end eliminados). Si `USE_RAG=True` se muestra el device_type de cada chunk recuperado desde `cisco_ios_xe_16.11.1_chunks.jsonl`.

In [ ]:
IDX = 0

row = df.iloc[IDX]
print('ID            : ' + str(row['id']))
print('n_lines       : ' + str(row['n_lines']))
print(chr(10) + '📌 REQUIREMENT:')
print(row['requirement'])
print(chr(10) + '🤖 PREDICCION:')
print(predictions[IDX])
print(chr(10) + '✅ GROUND TRUTH:')
print(references[IDX])
r1_val = compute_rouge([predictions[IDX]], [references[IDX]])['rouge1']
print(chr(10) + 'ROUGE-1 (normalizado): ' + '{:.4f}'.format(r1_val))
if USE_RAG and retrieved_logs[IDX]['chunks_retrieved']:
    print(chr(10) + '📚 Chunks recuperados:')
    for sid, dtype in zip(retrieved_logs[IDX]['chunks_retrieved'],
                          retrieved_logs[IDX]['chunk_device_types']):
        print('  [' + dtype + '] ' + sid)
    top = retrieved_logs[IDX]['top_score']
    if top is not None:
        print('  top_score: ' + str(top))